# DENTRACE — Level 6: 4-Class YOLOv8n-Seg Resumable Training on Google Colab T4
**Project:** DENTRACE — Local-First Dental OPG Anomaly Detection & Insurance Claim Pre-Authorization Engine  
**Institutional Framework:** Academic College Decision-Support System  
**Regulatory Notice:** *Draft for dentist review. Not a diagnosis. Academic decision-support prototype.*

### Target Pathology Classes (DENTEX Challenge Abnormalities):
0. `impacted_tooth` (Authoritative count: 604)
1. `caries` (Authoritative count: 2,189)
2. `periapical_lesion` (Authoritative count: 158)
3. `deep_caries` (Authoritative count: 578)

**Medical Semantics Notice:** Annotations represent whole abnormal tooth envelopes rather than exact cavity or lesion boundaries.
**Hardware Notice:** Optimized for Google Colab NVIDIA Tesla T4 (16 GB VRAM) with checkpoint auto-saving and resumability.

In [ ]:
# 1. Environment & Hardware Verification
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")

In [ ]:
# 2. Install Required Dependencies
!pip install -q "ultralytics>=8.3.0" onnx onnxruntime onnxruntime-tools albumentations

In [ ]:
# 3. Dataset Setup (Mount Google Drive or Upload Archive)
# Note: User approval is required before uploading any external dataset.
import os
from pathlib import Path

# Option A: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/dentex_yolo_seg.zip -d /content/dentex_dataset

# Option B: Local upload
!mkdir -p /content/dentex_dataset
# If uploaded directly to /content/dentex_yolo_seg.zip:
# !unzip -q /content/dentex_yolo_seg.zip -d /content/dentex_dataset/

# Generate data.yaml
yaml_content = """
path: /content/dentex_dataset
train: images/train
val: images/val

names:
  0: impacted_tooth
  1: caries
  2: periapical_lesion
  3: deep_caries

nc: 4
"""
with open('/content/dentex_dataset/data.yaml', 'w') as f:
    f.write(yaml_content.strip())
print("Configured /content/dentex_dataset/data.yaml")

In [ ]:
# 4. Launch Resumable 50-Epoch YOLOv8n-Seg Fine-Tuning
from ultralytics import YOLO

# Check if resuming from an interrupted checkpoint
resume_ckpt = '/content/runs/segment/dentex_yolov8n_seg/weights/last.pt'
if os.path.exists(resume_ckpt):
    print(f"Resuming training from checkpoint: {resume_ckpt}")
    model = YOLO(resume_ckpt)
    results = model.train(resume=True)
else:
    print("Starting new training run from yolov8n-seg.pt...")
    model = YOLO('yolov8n-seg.pt')
    results = model.train(
        data='/content/dentex_dataset/data.yaml',
        epochs=50,
        batch=16,                # T4 16GB supports batch=16 smoothly
        imgsz=640,
        device=0,
        seed=42,
        deterministic=True,
        patience=12,             # Early stopping patience
        save=True,
        save_period=5,           # Checkpoint saved every 5 epochs for resumability
        workers=4,
        half=True,               # Mixed precision AMP
        project='runs/segment',
        name='dentex_yolov8n_seg',
        exist_ok=True,
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        mosaic=0.5,              # Controlled augmentation preserving arch layout
        fliplr=0.5,
        flipud=0.0,              # Radiographs must never be flipped vertically
    )

In [ ]:
# 5. Class-Wise Validation & Mask Metric Audit
val_metrics = model.val(
    data='/content/dentex_dataset/data.yaml',
    split='val',
    device=0,
    batch=16,
    imgsz=640
)

print("\n=== VALIDATION METRICS (DENTEX Validation Split) ===")
print(f"Box Precision:  {val_metrics.box.mp:.4f}")
print(f"Box Recall:     {val_metrics.box.mr:.4f}")
print(f"Box mAP50:      {val_metrics.box.map50:.4f}")
print(f"Box mAP50-95:   {val_metrics.box.map:.4f}")
print(f"Mask Precision: {val_metrics.seg.mp:.4f}")
print(f"Mask Recall:    {val_metrics.seg.mr:.4f}")
print(f"Mask mAP50:     {val_metrics.seg.map50:.4f}")
print(f"Mask mAP50-95:  {val_metrics.seg.map:.4f}")

classes = ['impacted_tooth', 'caries', 'periapical_lesion', 'deep_caries']
for idx, cls_name in enumerate(classes):
    if idx < len(val_metrics.seg.maps):
        print(f"  Class {idx} ({cls_name:18s}) Mask mAP50-95: {val_metrics.seg.maps[idx]:.4f}")

In [ ]:
# 6. Export to ONNX (FP32) and Dynamic INT8 Quantization
best_weights = 'runs/segment/dentex_yolov8n_seg/weights/best.pt'
export_model = YOLO(best_weights)

# Export FP32 ONNX
onnx_path = export_model.export(format='onnx', imgsz=640, opset=17)
print(f"Exported ONNX: {onnx_path}")

# Dynamic INT8 Quantization via ONNX Runtime
from onnxruntime.quantization import quantize_dynamic, QuantType
int8_path = 'runs/segment/dentex_yolov8n_seg/weights/best_int8.onnx'
quantize_dynamic(
    model_input=onnx_path,
    model_output=int8_path,
    weight_type=QuantType.QUInt8,
)
print(f"Quantized Dynamic INT8 ONNX: {int8_path}")

In [ ]:
# 7. Checkpoint Packaging and Download for Local Integration
!zip -r /content/dentrace_level6_checkpoints.zip runs/segment/dentex_yolov8n_seg/weights/best.pt runs/segment/dentex_yolov8n_seg/weights/best.onnx runs/segment/dentex_yolov8n_seg/weights/best_int8.onnx
from google.colab import files
files.download('/content/dentrace_level6_checkpoints.zip')